# 06 - Error analysis

Inspect false positives and false negatives for the leading model.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 50)

In [2]:
from app.ml.preprocessing import get_processed_dataset, split_features_labels, train_test_split_data
from app.feature_engineering.extractor import FEATURE_NAMES
from app.ml.train import train_single_model, predict_with_probability
from app.ml.preprocessing import fit_scaler

frame = get_processed_dataset()
features, labels = split_features_labels(frame, FEATURE_NAMES)
x_train, x_test, y_train, y_test = train_test_split_data(features, labels)
scaler = fit_scaler(x_train, save_path=None)
outcome = train_single_model('random_forest', x_train, y_train, x_test, y_test, scaler)
outcome['payload']['metrics']

2026-09-26 13:15:12 | INFO     | app.ml.train | Training Random Forest


2026-09-26 13:15:18 | INFO     | app.ml.train | Random Forest -> accuracy 0.8983 | f1 0.8899 | fpr 0.0972 | 5.00s


{'accuracy': 0.8983050847457628,
 'precision': 0.8868184955141477,
 'recall': 0.8929812369701181,
 'f1_score': 0.8898891966759003,
 'specificity': 0.9028436018957346,
 'false_positive_rate': 0.0971563981042654,
 'false_negative_rate': 0.10701876302988186,
 'true_positive': 1285,
 'true_negative': 1524,
 'false_positive': 164,
 'false_negative': 154,
 'roc_auc': 0.960520898860122,
 'pr_auc': 0.9589448264087004}

## Locate the errors

In [3]:
urls_train, urls_test, _, _ = train_test_split_data(
    frame['url'].to_numpy().reshape(-1, 1), labels)
predictions = outcome['predictions']

false_positive = (y_test == 0) & (predictions == 1)
false_negative = (y_test == 1) & (predictions == 0)
print(f'False positives: {false_positive.sum()}  False negatives: {false_negative.sum()}')

False positives: 164  False negatives: 154


In [4]:
print('Sample false positives (legitimate flagged as phishing):')
for url in urls_test[false_positive].ravel()[:10]:
    print(' ', url)

Sample false positives (legitimate flagged as phishing):
  https://secure.secure.works.ga:3000
  http://verification.cdn.technimbusmaple.io
  https://suspended.wkuixjd376kezuge.org/secure/posts/products?q=mxp
  https://mj9nkvu3sst.co/payment/status/reference.html?session=stshdt907b1o3zzeqq&redirect=account&id=8904466&email=3g1tof7%40mail.com
  http://confirm.orbitcedarbridge.in?session=mjfedg7r30ezn&ref=qfmwwm&email=b4c4c4a1o%40mail.com
  https://v3djqgdv2okazl.com/legal/support/suspended/support?id=548678&email=9z4vam8mn5%40mail.com&session=c8x1cl28xdlixd
  https://suspended.shopstudio.site/payment/auth/99ovywg3zl/0syy8pq1br26tje/gyjkwp0r88i.aspx
  https://security.account.solutions528.org
  https://blog.ve0pr8u6cxn.online/signin/lkmeab6zyt7zswwqwg?email=lmtbjj1v8%40mail.com&session=yukbt38ej4fqfdsx47hmx
  http://auth.6iwj7j7hkj7e.net


In [5]:
print('Sample false negatives (phishing flagged as legitimate):')
for url in urls_test[false_negative].ravel()[:10]:
    print(' ', url)

Sample false negatives (phishing flagged as legitimate):
  https://confirm.swiftshop.com/95cs529o79s8g0so8g?id=1089762&ref=jqthelysxc3&redirect=index
  http://shop.summittech.org/7pyleout
  https://id.digitalnorth.site:8080/secure/qsm09dwjhjui.htm?redirect=home&email=7hktpulgs5%40mail.com
  https://update.dropbox.com/unlock?ref=cabsz&utm_source=ufk89&email=s4kmst%40mail.com&id=1200290
  https://blog.bridge-systems-prime.co/2dnjzuzq9aj9gj3k
  https://confirm.slack.com/grm9i34waek2loq/wz35vqpe73m/contact/3mbrnr4sakn4ib4e
  https://api.amazoninc.com
  https://steamcare.in/suspended
  https://shopdeltadelta.icu/docs/signin
  https://systems.io?utm_source=5f1vp8&email=yxez4u4k7j%40mail.com&id=8592632
